# 🔬 CancerNet — A100 GPU Training
### Amal Raju

**EfficientNet-B4 + ViT-Base → Cross-Attention Fusion → IDC Classification**

---

## 1️⃣ GPU Check

In [ ]:
import torch

assert torch.cuda.is_available(), '❌ No GPU! → Runtime → Change runtime type → A100'

gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ {gpu} | {mem:.0f} GB | CUDA {torch.version.cuda} | PyTorch {torch.__version__}')
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free --format=csv,noheader

## 2️⃣ Upload & Setup Project

In [ ]:
import os
from google.colab import files

print('📁 Upload cancer-detection zip file...')
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
print(f'✅ {zip_file} ({len(uploaded[zip_file])/1e6:.1f} MB)')

!unzip -q -o "{zip_file}" -d /content/
os.chdir('/content/cancer-detection')
print(f'✅ Working dir: {os.getcwd()}')

## 3️⃣ Install Dependencies

In [ ]:
import torch
TV = torch.__version__.split('+')[0]
CV = torch.version.cuda.replace('.', '')

!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TV}+cu{CV}.html 2>/dev/null
!pip install -q torch-geometric timm albumentations omegaconf grad-cam scikit-image 2>/dev/null

import timm, albumentations, omegaconf
print(f'✅ timm={timm.__version__} | albumentations={albumentations.__version__} | omegaconf={omegaconf.__version__}')
try:
    import torch_geometric; print(f'✅ torch_geometric={torch_geometric.__version__}')
except:
    print('⚠️ torch_geometric unavailable — GNN branch will be skipped')

## 4️⃣ Download Dataset

In [ ]:
from google.colab import files as f
print('📁 Upload kaggle.json...')
f.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle configured')

In [ ]:
%%time
!pip install -q kaggle
os.makedirs('data/raw', exist_ok=True)

print('📥 Downloading dataset...')
!kaggle datasets download -d paultimothymooney/breast-histopathology-images -p data/ --unzip

# Fix path structure
if os.path.exists('data/IDC_regular_ps50_idx5'):
    !mv data/IDC_regular_ps50_idx5/* data/raw/
    !rm -rf data/IDC_regular_ps50_idx5
# If patient folders landed directly in data/
import glob
patient_dirs = glob.glob('data/[0-9]*')
if patient_dirs and not os.listdir('data/raw'):
    for d in patient_dirs:
        !mv "{d}" data/raw/

!echo "Patients:" && ls data/raw/ | wc -l
!du -sh data/raw/

## 5️⃣ Verify Dataset

In [ ]:
!python scripts/preprocess.py --config configs/config.yaml --verify-only

## 6️⃣ A100 Performance Tuning

This cell applies runtime optimizations specific to A100 hardware.

In [ ]:
# A100-specific optimizations applied at code level:
# ✅ GNN disabled (SLIC superpixel was the #1 CPU bottleneck)
# ✅ ElasticTransform removed (slow CPU augmentation)
# ✅ image_np skipped when GNN off (no wasted cv2.resize)
# ✅ torch.compile() enabled (fused GPU kernels)
# ✅ TF32 + cuDNN benchmark (Ampere-native fast math)
# ✅ persistent_workers + prefetch_factor=4
# ✅ non_blocking CUDA transfers
# ✅ batch_size=128, num_workers=4

print('Config:')
!cat configs/config.yaml

## 7️⃣ Train 🚀

In [ ]:
%%time

CONFIG = 'configs/config.yaml'
!python scripts/train.py --config {CONFIG}

## 8️⃣ Evaluate

In [ ]:
CONFIG = 'configs/config.yaml'
!ls -lh checkpoints/

In [ ]:
# Standard evaluation
!python scripts/evaluate.py --checkpoint checkpoints/best_model.pth --config {CONFIG}

In [ ]:
# TTA evaluation
!python scripts/evaluate.py --checkpoint checkpoints/best_model.pth --config {CONFIG} --tta

In [ ]:
# Grad-CAM
!python scripts/evaluate.py --checkpoint checkpoints/best_model.pth --config {CONFIG} --gradcam --n-gradcam 10

## 📊 Results

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from pathlib import Path

# ROC
import os
if os.path.exists('results/roc_curve.png'):
    print('📈 ROC Curve:')
    display(IPImage(filename='results/roc_curve.png'))

# Grad-CAM grid
gc_dir = Path('results/gradcam')
if gc_dir.exists():
    imgs = sorted(gc_dir.glob('*.png'))[:10]
    if imgs:
        cols = min(5, len(imgs))
        rows = (len(imgs) + cols - 1) // cols
        fig, ax = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        ax = ax.flatten() if len(imgs) > 1 else [ax]
        for i, p in enumerate(imgs):
            ax[i].imshow(plt.imread(str(p))); ax[i].set_title(p.stem, fontsize=8); ax[i].axis('off')
        for i in range(len(imgs), len(ax)): ax[i].axis('off')
        plt.suptitle('Grad-CAM', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 📈 Model Info

In [ ]:
import sys; sys.path.insert(0, '/content/cancer-detection')
from omegaconf import OmegaConf
from src.models.cancernet import CancerNet

cfg = OmegaConf.load('configs/config.yaml')
model = CancerNet(cfg)
params = model.count_parameters()
total = sum(params.values())
print('Trainable Parameters:')
for k, v in params.items(): print(f'  {k:15s}: {v:>12,} ({v/total*100:.1f}%)')
print(f'  {"TOTAL":15s}: {total:>12,}')

## 9️⃣ Download Results

In [ ]:
!zip -r /content/cancernet_results.zip checkpoints/ results/ logs/ -x '*.pyc' '__pycache__/*'
!ls -lh /content/cancernet_results.zip

from google.colab import files
files.download('/content/cancernet_results.zip')

In [ ]:
# Optional: Save to Google Drive
SAVE_TO_DRIVE = False
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    d = '/content/drive/MyDrive/CancerNet_Results'
    os.makedirs(d, exist_ok=True)
    !cp -r checkpoints/ "{d}/" && cp -r results/ "{d}/" && cp -r logs/ "{d}/"
    print(f'✅ Saved to {d}')